# three.ws GPU quickstart: image to 3D on your NVIDIA GPU

This notebook runs the three.ws self-hosted GPU stack on the machine in front of you:

1. **TripoSR** (MIT licence, Tripo AI and Stability AI) turns one photo into a textured 3D mesh in a few seconds.
2. The **three.ws inference node** proves your GPU can serve the open inference network, and can optionally join it.

Everything runs locally. The model weights are baked into the container, so no API key and no download is needed. Run the cells top to bottom (Shift+Enter). A browser version of step 1 lives at the **studio** link (port 8000).

## 1. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA runtime", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())

## 2. Wait for the GPU engine

The studio server in this container loads the 1.7 GB TripoSR checkpoint onto the GPU once, at startup. Every cell below reuses that copy instead of loading a second one, so the notebook itself holds no CUDA memory.

In [ ]:
import three_ws_brev as tw
from IPython.display import HTML, display

health = tw.wait_until_ready()
info = tw.system()
engine = info["engine"]
print(f"{engine['gpu_name']} (sm_{engine['compute_capability'].replace('.', '')}, {engine['gpu_memory_mb']:,} MB)")
print(f"driver {info['nvidia_smi'].get('driver_version')} | torch {engine['torch']} | CUDA {engine['cuda_runtime']}")
print(f"TripoSR load to GPU: {engine['model_load_ms']:.0f} ms | U2-Net session: {engine['matting_load_ms']:.0f} ms on {engine['matting_providers'][0]}")
print(f"container start to ready: {health['startup_to_ready_s']} s")

## 3. Generate a 3D model

Pick one of the bundled samples, or set `IMAGE_PATH` to a PNG, JPEG or WEBP you uploaded with the JupyterLab file browser (drag a file onto the left panel). Best results: one object, plain background, even light.

In [ ]:
print("samples:", ", ".join(tw.samples()))
SAMPLE = "chair.png"
IMAGE_PATH = None  # e.g. "/workspace/my-photo.jpg"

record = tw.generate(sample=None if IMAGE_PATH else SAMPLE, image_path=IMAGE_PATH, resolution=256)
display(HTML(tw.timings_table(record)))
print("saved:", "/workspace/outputs/" + record["id"] + ".glb")

## 4. Look at it

Drag to orbit, scroll to zoom. The mesh is a standard binary glTF (GLB) with baked vertex colours, so it opens in Blender, three.js, Unity, Unreal and NVIDIA Omniverse unchanged. Download it from `/workspace/outputs/` in the file browser.

In [ ]:
display(HTML(tw.viewer_html(tw.fetch_glb(record))))

## 5. Where the milliseconds go

Every timing above is measured after `torch.cuda.synchronize()`, so it is finished GPU work, not queued work.

| Stage | What runs on the GPU |
|---|---|
| Background removal | U2-Net matting on ONNX Runtime's CUDA execution provider, then the subject is recentred at 85% of the frame on mid grey. |
| TripoSR forward pass | A DINO ViT-B/16 image encoder and a transformer decode the photo into a triplane: a compact 3D representation of shape and colour. This is the stage that scales with GPU compute. |
| Marching cubes | The triplane is queried on a 256^3 grid in 8,192-point chunks and `torchmcubes` extracts the surface with a CUDA kernel. Raising the resolution to 320 roughly doubles this stage. |
| GLB export | CPU-side: orientation fix and binary glTF serialisation by trimesh. |

The first generation after a cold start also pays one-time CUDA kernel selection. The benchmark below runs warm.

In [ ]:
from studio.core import summarize_runs

RUNS = 5
reports = [tw.generate(sample=SAMPLE, resolution=256)["timings"] for _ in range(RUNS)]
display(HTML(tw.benchmark_table(summarize_runs(reports))))

## 6. The three.ws inference node

The [three.ws open inference network](https://github.com/nirholas/three.ws/blob/main/docs/inference-node-operator.md) lets independently operated GPUs serve agent inference jobs. Each node has an ed25519 identity and signs every result, so anyone can verify who computed what.

The node client is installed in this container. Its self-test runs the network's proof workload (`all-MiniLM-L6-v2`, Apache-2.0, fp32 on ONNX Runtime CUDA) and reports the device that executed it.

In [ ]:
node = tw.node_self_test()
status = "OK" if node["ok"] else "FAILED"
print(f"self-test {status}: {node['model']} -> {node['dimensions']} dims on {node['device']} ({node['dtype']}) in {node['elapsed_ms']} ms")
print("node identity (public key):", node.get("public_key"))

### Optional: join the network

Set `CONNECT = True` to register this node with three.ws. Registration publishes only the public key above and a label; the secret key stays in `/workspace/identity/node-identity.json` on this machine. To keep serving jobs after this notebook closes, restart the container with `THREE_WS_CONNECT_NODE=1` (see the README).

In [ ]:
import os
CONNECT = False
NODE_LABEL = "brev-quickstart"

if CONNECT:
    env = {**os.environ, "DEVICE": "cuda", "NODE_LABEL": NODE_LABEL,
           "IDENTITY_PATH": "/workspace/identity/node-identity.json"}
    out = subprocess.run(["node", "src/index.js", "--register-only"], cwd="/opt/node-operator",
                         env=env, capture_output=True, text=True, timeout=300)
    print(out.stdout[-3000:], out.stderr[-3000:])
else:
    print("Skipped. Set CONNECT = True and re-run this cell to register the node.")

## Next steps

- Turn a generated mesh into a rigged, animated avatar: https://three.ws/forge
- Run the node full time: https://github.com/nirholas/three.ws/blob/main/docs/inference-node-operator.md
- How this Launchable is built: https://github.com/nirholas/three.ws/tree/main/distributions/nvidia-brev

three.ws is a member of the NVIDIA Inception program.